# Mission 06: Agent Assembly & Production Server Registration

이 노트북은 지금까지 개별 단계로 학습하고 구현한 모든 에이전트 핵심 부품들을 하나로 조립하여, 실제 FastAPI 프로덕션 서버에 연동 가능한 완성형 에이전트(`harness_agent.py`)를 어셈블리하고 등록해보는 실습입니다.

난이도가 높은 과정이므로 본 노트북의 가이드라인 코드를 꼼꼼히 확인하고 주피터 환경에서 사전 검증을 진행하세요.

In [ ]:
# 1. 환경 변수 및 패키지 탐색 경로 로드
import sys
import os
from dotenv import load_dotenv

while not os.path.exists("app") and os.getcwd() != "/":
    os.chdir("..")
sys.path.append(os.path.abspath("src"))
sys.path.append(os.path.abspath("."))
load_dotenv(override=True)

print(f"📌 현재 작업 디렉토리: {os.getcwd()}")

### [단계 1] configs 파일 로드 및 AgentContext 설정

`configs/logging.config` 파일에서 실시간 로깅 모니터링 활성화 여부(`logging_enabled`)를 인출하여 `AgentContext`에 바인딩합니다.

In [ ]:
import json
from app.utils.context import AgentContext

# 1. logging.config 로드
config_path = "configs/logging.config"
logging_enabled = False
log_path = "./artifacts/agent_audit_trail.json"

if os.path.exists(config_path):
    with open(config_path, "r", encoding="utf-8") as f:
        try:
            cfg = json.load(f)
            logging_enabled = cfg.get("logging_enabled", False)
            log_path = cfg.get("log_path", "./artifacts/agent_audit_trail.json")
        except Exception as e:
            print(f"⚠️ 설정 파일 로드 에러: {e}")

print(f"📊 Config 로깅 활성화 상태: {logging_enabled} | 로그 경로: {log_path}")

# 2. AgentContext 인스턴스 초기화
context_obj = AgentContext(
    logging_enabled=logging_enabled,
    log_path=log_path,
    response_mode="chat",
    hitl_enabled=False,
    debug_mode=False
)
print("✅ AgentContext 생성 완료!")

### [단계 2] 5대 핵심 부품 결합 및 harness_agent 빌드

1. **LLM**: Gemini-3.5-Flash 호출 엔진
2. **SQLite Saver**: 영속성 체크포인터 메모리 (`checkpoints.db` 연동)
3. **Tools**: 범용 8대 도구 세트 (`tools_chatbot` 바인딩)
4. **Logging Middleware**: 실시간 로그 및 감사 추적 파일 기록기
5. **Dynamic Prompt Middleware**: 5계층 프롬프트 및 SQLite RAG 기억 탐색 주입기

In [ ]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain.agents import create_agent
from app.utils import get_llm
from app.tools import tools_chatbot
from app.middleware.logging_middleware import LoggingMiddleware
from app.prompts import harness_agent_prompt_middleware

def get_agent_executor():
    # 1. Gemini 모델 기동
    llm = get_llm(model_name="google_vertexai:gemini-3.5-flash", temperature=0.0)
    
    # 2. SQLite 기반 Working Memory 체크포인터 연결
    conn = sqlite3.connect("app/database/checkpoints.db", check_same_thread=False)
    memory = SqliteSaver(conn)
    
    # 3. 범용 도구 바인딩
    tools = tools_chatbot
    
    # 4. 모니터링 로깅 미들웨어
    logging_middleware = LoggingMiddleware(log_dir="./artifacts/logs")
    
    # 5. 미들웨어 체인 구성 (로깅 우선 격발 -> 프롬프트 동적 주입)
    middleware = [logging_middleware, harness_agent_prompt_middleware]
    
    # 6. 에이전트 빌드
    harness_agent = create_agent(
        model=llm,
        tools=tools,
        system_prompt=None,  # 동적 프롬프트 미들웨어가 5계층을 구성하므로 None 처리
        checkpointer=memory,
        middleware=middleware,
        context_schema=AgentContext
    )
    return harness_agent

agent_executor = get_agent_executor()
print("🚀 완성형 Harness Agent 빌드 완료!")

### [단계 3] 테스트 호출 및 에이전트 동작 검증

미들웨어들이 백그라운드에서 SQLite RAG 장기기억 탐색과 로깅 처리를 완전 자동 수행하는지 시뮬레이션 테스트를 구동합니다.

In [ ]:
# configs에서 로깅을 켜고 동작 확인
context_obj.logging_enabled = True

inputs = {"messages": [{"role": "user", "content": "check_server_status 툴로 상태를 체크하고 deploy_log_audit으로 감사해 줘."}]}
config = {"configurable": {"thread_id": "assembly_test_session_01"}}

print("🔄 에이전트 런타임 호출 시작...")
result = agent_executor.invoke(inputs, config=config, context=context_obj)

messages = result.get("messages", [])
if messages:
    print("\n🤖 에이전트 최종 답변:")
    print("="*60)
    print(messages[-1].content)
    print("="*60)
else:
    print("❌ 에이전트 응답을 받지 못했습니다.")

## 📌 Part 2. 프로덕션 서버 등록 및 실구동 포팅 절차

노트북에서 안정적인 빌드를 검증했다면, 이제 실제 FastAPI 웹 애플리케이션 서버로 해당 에이전트를 등록해야 합니다. 다음 절차를 차근차근 밟으세요.

### 1단계. `app/agents/harness_agent.py` 파일 생성 및 포팅
*   위의 **[단계 2]**에서 정상 동작한 코드를 [app/agents/harness_agent.py](file:///mnt/c/Users/hyoun/Desktop/harness_agent/app/agents/harness_agent.py)에 그대로 덮어쓰기하여 저장합니다.

### 2단계. `app/agents/__init__.py` 에이전트 레지스트리 점검
*   [app/agents/__init__.py](file:///mnt/c/Users/hyoun/Desktop/harness_agent/app/agents/__init__.py) 에이전트 등록소 파일을 열고, `AGENT_REGISTRY` 안에 `harness_agent` 설정이 올바르게 잡혀있는지 점검합니다:
```python
AGENT_REGISTRY = [
    # ...
    {
        "name": "harness_agent",
        "module": "app.agents.harness_agent",
        "prefix": "/harness_agent",
        "tags": ["HarnessAgent"],
        "description": "수강생들이 단계별 실무 미션을 통해 완성해나가는 실습용 에이전트"
    }
]
```
*   서버 기동 시, 레지스트리에 등록된 `module` 패키지로부터 `agent_executor` 인스턴스를 자동으로 임포트하여 라우팅합니다.

### 3단계. FastAPI 웹 서버 구동
*   터미널에서 아래 명령을 실행해 에이전트 웹 서버를 가동합니다:
```bash
uvicorn app.server:app --reload --port 8000
```
*   서버가 로드될 때 터미널 콘솔 로그에 `✅ Registered agent: harness_agent at /harness_agent`가 찍히는지 확인하세요!